# 🔬 NEXUS Stock AI — Phases 6, 10, 11, 12, 13, & 14
## Unified Multi-Modal Feature Engineering & Target Construction

**Objective:** Build the definitive, leak-free machine learning training dataset (`model_dataset_v1.parquet`) by fusing technical price indicators with daily aggregated FinBERT sentiment signals across our 10 MVP tickers.

### 📋 Execution Plan
1. **Phase 6 (Technical Feature Engineering):** Compute returns (1d, 3d, 5d), moving averages (SMA 5/20/50, EMA 12/26), momentum (RSI-14, MACD, MACD Signal), volatility (5d, 20d), and volume metrics per ticker.
2. **Phase 10 (Daily News Sentiment Aggregation):** Group `sentiment_news.parquet` by `[Stock_symbol, trade_date_target]` to calculate volume (`news_count`), distributions (`sentiment_mean`, `std`, `min`, `max`), and polarity ratios (`positive_ratio`, `negative_ratio`, `neutral_ratio`).
3. **Phases 11 & 13 (Table Fusion & Imputation):** Left-join technical prices with daily sentiment; impute neutral values (`0.0`) and zero news count for days without news coverage.
4. **Phase 12 (Next-Day Target Formulation):** Vectorized calculation of next-day binary direction:
   $$\text{target}_{t} = \mathbb{I}(\text{close}_{t+1} > \text{close}_{t}) \in \{0, 1\}$$
5. **Data Cleanup & Leakage Prevention:** Discard warm-up rows (first 49 trading days per ticker where `sma_50` is NaN) and terminal trading records where target direction is undefined.
6. **Phase 14 (Persistence & Metadata):** Persist `model_dataset_v1.parquet`, generate `dataset_metadata.json`, update `nexus_data_backup.zip`, and render diagnostic validation tables.

### ⚙️ Step 1: Environment Setup & Google Drive Mount

In [1]:
import os
import sys
import gc
import time
import json
import shutil
import zipfile
import pandas as pd
import numpy as np

# Mount Google Drive if in Colab
try:
    from google.colab import drive
    print("Mounting Google Drive at /content/drive...")
    drive.mount("/content/drive")
    print("✓ Google Drive mounted successfully.")
except Exception as e:
    print(f"Drive mount note: {e}")

GDRIVE_DIR = "/content/drive/MyDrive/NEXUS_Stock_AI/data"
LOCAL_DIR = "./data"

def resolve_path(filename):
    gdrive_p = os.path.join(GDRIVE_DIR, filename)
    local_p = os.path.join(LOCAL_DIR, filename)
    if os.path.exists(gdrive_p):
        return gdrive_p
    elif os.path.exists(local_p):
        return local_p
    return gdrive_p

PRICES_INPUT = resolve_path("cleaned_prices.parquet")
SENTIMENT_INPUT = resolve_path("sentiment_news.parquet")

print(f"Cleaned Prices Path:    {PRICES_INPUT} (Exists: {os.path.exists(PRICES_INPUT)})")
print(f"Sentiment News Path:    {SENTIMENT_INPUT} (Exists: {os.path.exists(SENTIMENT_INPUT)})")

Mounting Google Drive at /content/drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted successfully.
Cleaned Prices Path:    /content/drive/MyDrive/NEXUS_Stock_AI/data/cleaned_prices.parquet (Exists: True)
Sentiment News Path:    /content/drive/MyDrive/NEXUS_Stock_AI/data/sentiment_news.parquet (Exists: True)


### 📈 Step 2: Phase 6 — Technical Feature Engineering
Calculate technical signals grouped strictly by `stock_symbol` to prevent cross-ticker leakage:
- **Returns:** `return_1d`, `return_3d`, `return_5d`
- **Moving Averages:** `sma_5`, `sma_20`, `sma_50`, `ema_12`, `ema_26`
- **Momentum:** Wilder's `rsi_14`, `macd`, `macd_signal`
- **Volatility:** `volatility_5d`, `volatility_20d` (rolling std dev of 1-day returns)
- **Volume:** `volume_change_1d`, `volume_sma_20`

In [2]:
print("=" * 80)
print("PHASE 6: VECTORIZED TECHNICAL FEATURE ENGINEERING")
print("=" * 80)
t0 = time.time()

prices_df = pd.read_parquet(PRICES_INPUT)
prices_df['date'] = pd.to_datetime(prices_df['date']).dt.normalize()
prices_df = prices_df.sort_values(['stock_symbol', 'date']).reset_index(drop=True)
print(f"Loaded {len(prices_df):,} cleaned price records across {prices_df['stock_symbol'].nunique()} tickers.")

def compute_ticker_technical_features(df_grp):
    df = df_grp.copy()
    close = df['close']
    volume = df['volume']
    
    # 1. Returns
    df['return_1d'] = close.pct_change(1)
    df['return_3d'] = close.pct_change(3)
    df['return_5d'] = close.pct_change(5)
    
    # 2. Moving Averages
    df['sma_5'] = close.rolling(window=5).mean()
    df['sma_20'] = close.rolling(window=20).mean()
    df['sma_50'] = close.rolling(window=50).mean()
    df['ema_12'] = close.ewm(span=12, adjust=False).mean()
    df['ema_26'] = close.ewm(span=26, adjust=False).mean()
    
    # 3. Momentum: MACD & MACD Signal
    df['macd'] = df['ema_12'] - df['ema_26']
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    
    # Momentum: Wilder's RSI 14
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    df['rsi_14'] = 100 - (100 / (1 + rs))
    df['rsi_14'] = df['rsi_14'].fillna(50.0)
    
    # 4. Volatility (Rolling Std Dev of 1-Day Returns)
    df['volatility_5d'] = df['return_1d'].rolling(window=5).std()
    df['volatility_20d'] = df['return_1d'].rolling(window=20).std()
    
    # 5. Volume Features
    df['volume_change_1d'] = volume.pct_change(1)
    df['volume_sma_20'] = volume.rolling(window=20).mean()
    
    return df

engineered_prices = prices_df.groupby('stock_symbol', group_keys=False).apply(compute_ticker_technical_features)
print(f"✓ Engineered technical features for {len(engineered_prices):,} rows in {time.time() - t0:.2f}s")
display(engineered_prices[['stock_symbol', 'date', 'close', 'return_1d', 'sma_50', 'rsi_14', 'macd']].tail(3))

PHASE 6: VECTORIZED TECHNICAL FEATURE ENGINEERING
Loaded 67,315 cleaned price records across 9 tickers.
✓ Engineered technical features for 67,315 rows in 0.25s


/tmp/ipykernel_16096/2296360180.py:52: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  engineered_prices = prices_df.groupby('stock_symbol', group_keys=False).apply(compute_ticker_technical_features)


,stock_symbol,date,close,return_1d,sma_50,rsi_14,macd
67312,TSLA,2023-12-26,256.609985,0.016116,232.662800,60.824035,5.495890
67313,TSLA,2023-12-27,261.440002,0.018822,232.813200,63.793639,6.031711
67314,TSLA,2023-12-28,253.179993,-0.031594,232.779799,55.978815,5.723858


### 📰 Step 3: Phase 10 — Aggregate Daily News Sentiment
Aggregate FinBERT scores by `[Stock_symbol, trade_date_target]`:
- `news_count`: Volume of news articles targeting this session
- `sentiment_mean`: Average sentiment score
- `sentiment_std`: Dispersion of sentiment (filled with `0.0` for single-article days)
- `sentiment_min`, `sentiment_max`: Range of sentiment
- `positive_ratio`, `negative_ratio`, `neutral_ratio`: Distribution of sentiment polarity

In [3]:
print("=" * 80)
print("PHASE 10: AGGREGATE DAILY NEWS SENTIMENT")
print("=" * 80)
t1 = time.time()

news_df = pd.read_parquet(SENTIMENT_INPUT)
print(f"Loaded {len(news_df):,} FinBERT sentiment records.")

# Normalize merge date and ticker casing
news_df['trade_date_target'] = pd.to_datetime(news_df['trade_date_target']).dt.normalize()
news_df['Stock_symbol'] = news_df['Stock_symbol'].astype(str).str.strip().str.upper()

# Polarity masks
news_df['is_pos'] = (news_df['sentiment_score'] > 0.05).astype(float)
news_df['is_neg'] = (news_df['sentiment_score'] < -0.05).astype(float)
news_df['is_neu'] = (news_df['sentiment_score'].abs() <= 0.05).astype(float)

# Grouped daily aggregation
sentiment_agg = news_df.groupby(['Stock_symbol', 'trade_date_target']).agg(
    news_count=('sentiment_score', 'count'),
    sentiment_mean=('sentiment_score', 'mean'),
    sentiment_std=('sentiment_score', 'std'),
    sentiment_min=('sentiment_score', 'min'),
    sentiment_max=('sentiment_score', 'max'),
    pos_items=('is_pos', 'sum'),
    neg_items=('is_neg', 'sum'),
    neu_items=('is_neu', 'sum')
).reset_index()

# Fill std with 0 for single-article trading days
sentiment_agg['sentiment_std'] = sentiment_agg['sentiment_std'].fillna(0.0)

# Compute exact ratios
sentiment_agg['positive_ratio'] = sentiment_agg['pos_items'] / sentiment_agg['news_count']
sentiment_agg['negative_ratio'] = sentiment_agg['neg_items'] / sentiment_agg['news_count']
sentiment_agg['neutral_ratio'] = sentiment_agg['neu_items'] / sentiment_agg['news_count']

# Drop intermediate counts
sentiment_agg = sentiment_agg.drop(columns=['pos_items', 'neg_items', 'neu_items'])

print(f"✓ Aggregated into {len(sentiment_agg):,} ticker-date sentiment sessions in {time.time() - t1:.2f}s")
display(sentiment_agg.head(3))

PHASE 10: AGGREGATE DAILY NEWS SENTIMENT
Loaded 61,992 FinBERT sentiment records.
✓ Aggregated into 6,750 ticker-date sentiment sessions in 1.41s


,Stock_symbol,trade_date_target,news_count,sentiment_mean,sentiment_std,sentiment_min,sentiment_max,positive_ratio,negative_ratio,neutral_ratio
0,AAPL,2020-03-09,3,-0.365338,0.488528,-0.925281,-0.026141,0.000000,0.666667,0.333333
1,AAPL,2020-03-10,8,-0.378180,0.452301,-0.947825,0.081098,0.125000,0.625000,0.250000
2,AAPL,2020-03-11,14,-0.262766,0.650021,-0.965545,0.913894,0.357143,0.500000,0.142857


### 🔗 Step 4: Phases 11 & 13 — Merge Datasets & Impute Days Without News
- Left-join prices with sentiment on `stock_symbol == Stock_symbol` and `date == trade_date_target`.
- Trading days with no articles: `news_count = 0` and all sentiment metrics = `0.0`.

In [4]:
print("=" * 80)
print("PHASES 11 & 13: LEFT JOIN & MISSING VALUE IMPUTATION")
print("=" * 80)
t2 = time.time()

merged_df = engineered_prices.merge(
    sentiment_agg,
    left_on=['stock_symbol', 'date'],
    right_on=['Stock_symbol', 'trade_date_target'],
    how='left'
)

# Impute zero-sentiment for trading days without news coverage
merged_df['news_count'] = merged_df['news_count'].fillna(0).astype(int)
senti_metrics = [
    'sentiment_mean', 'sentiment_std', 'sentiment_min', 'sentiment_max',
    'positive_ratio', 'negative_ratio', 'neutral_ratio'
]
for col in senti_metrics:
    merged_df[col] = merged_df[col].fillna(0.0).astype(np.float32)

# Clean up auxiliary merge columns
merged_df = merged_df.drop(columns=['Stock_symbol', 'trade_date_target'], errors='ignore')

days_with_news = (merged_df['news_count'] > 0).sum()
print(f"Total Trading Sessions:    {len(merged_df):,}")
print(f"Sessions With News:        {days_with_news:,} ({days_with_news/len(merged_df)*100:.1f}%)")
print(f"Sessions Without News (0): {len(merged_df) - days_with_news:,} (Imputed to 0.0)")
print(f"✓ Merged and imputed in {time.time() - t2:.2f}s")

PHASES 11 & 13: LEFT JOIN & MISSING VALUE IMPUTATION
Total Trading Sessions:    67,315
Sessions With News:        6,750 (10.0%)
Sessions Without News (0): 60,565 (Imputed to 0.0)
✓ Merged and imputed in 0.05s


### 🎯 Step 5: Phase 12 — Target Creation & Leakage-Free Trimming
- **Binary Direction Target:** `(close_{t+1} > close_{t}).astype(int)` per ticker.
- **Warm-up Period Removal:** Drop initial rows where `sma_50` is NaN (first 49 trading days per ticker).
- **Terminal Row Removal:** Drop the final row of each ticker where next-day close is unknown.

In [5]:
print("=" * 80)
print("PHASE 12: TARGET CREATION & WARMUP TRIMMING")
print("=" * 80)
t3 = time.time()

# Vectorized next-day close per ticker
next_close = merged_df.groupby('stock_symbol')['close'].shift(-1)
merged_df['target'] = (next_close > merged_df['close']).astype(int)

# Leakage prevention & cleanup:
# 1. Drop rows where next_close is NaN (terminal row per ticker)
# 2. Drop rows where sma_50 is NaN (warm-up period of 49 trading days)
valid_mask = next_close.notna() & merged_df['sma_50'].notna()
clean_model_df = merged_df[valid_mask].copy().reset_index(drop=True)

# Ensure volume_change_1d and volatility_5d leading nulls are cleared
clean_model_df = clean_model_df.dropna().reset_index(drop=True)

dropped_rows = len(merged_df) - len(clean_model_df)
print(f"Raw Merged Rows:         {len(merged_df):,}")
print(f"Dropped Warmup/Terminal: {dropped_rows:,} rows")
print(f"Final Clean Rows:        {len(clean_model_df):,}")
print(f"Remaining Null Values:   {clean_model_df.isna().sum().sum()} (Strictly 0)")
print(f"✓ Completed in {time.time() - t3:.2f}s")

PHASE 12: TARGET CREATION & WARMUP TRIMMING
Raw Merged Rows:         67,315
Dropped Warmup/Terminal: 450 rows
Final Clean Rows:        66,865
Remaining Null Values:   0 (Strictly 0)
✓ Completed in 0.06s


### 💾 Step 6: Phase 14 — Persistence, Metadata Generation, & ZIP Backup

In [6]:
print("=" * 80)
print("PHASE 14: PERSISTENCE & METADATA")
print("=" * 80)

target_out_dir = GDRIVE_DIR if os.path.exists(GDRIVE_DIR) else LOCAL_DIR
os.makedirs(target_out_dir, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

# 1. Save model_dataset_v1.parquet
output_parquet = os.path.join(target_out_dir, "model_dataset_v1.parquet")
local_parquet = os.path.join(LOCAL_DIR, "model_dataset_v1.parquet")
clean_model_df.to_parquet(output_parquet, engine='pyarrow', compression='zstd', index=False)

if os.path.abspath(output_parquet) != os.path.abspath(local_parquet):
    shutil.copy2(output_parquet, local_parquet)
print(f"✓ Saved dataset: {output_parquet} ({os.path.getsize(output_parquet)/(1024*1024):.2f} MB)")

# 2. Generate dataset_metadata.json
feature_cols = [
    'open', 'high', 'low', 'close', 'adj_close', 'volume',
    'return_1d', 'return_3d', 'return_5d',
    'sma_5', 'sma_20', 'sma_50', 'ema_12', 'ema_26',
    'macd', 'macd_signal', 'rsi_14',
    'volatility_5d', 'volatility_20d',
    'volume_change_1d', 'volume_sma_20',
    'news_count', 'sentiment_mean', 'sentiment_std',
    'sentiment_min', 'sentiment_max',
    'positive_ratio', 'negative_ratio', 'neutral_ratio'
]

class_0_count = int((clean_model_df['target'] == 0).sum())
class_1_count = int((clean_model_df['target'] == 1).sum())

metadata = {
    "dataset_name": "NEXUS_Stock_AI_model_dataset_v1",
    "total_rows": len(clean_model_df),
    "total_columns": len(clean_model_df.columns),
    "feature_count": len(feature_cols),
    "feature_columns": feature_cols,
    "target_column": "target",
    "target_classes": {"0": "DOWN or FLAT", "1": "UP"},
    "class_balance": {
        "class_0_down": class_0_count,
        "class_1_up": class_1_count,
        "class_1_percentage": round(class_1_count / len(clean_model_df) * 100, 2)
    },
    "date_ranges": {
        "start_date": clean_model_df['date'].min().strftime('%Y-%m-%d'),
        "end_date": clean_model_df['date'].max().strftime('%Y-%m-%d')
    },
    "ticker_breakdown": clean_model_df['stock_symbol'].value_counts().to_dict()
}

meta_path_drive = os.path.join(target_out_dir, "dataset_metadata.json")
meta_path_local = os.path.join(LOCAL_DIR, "dataset_metadata.json")
with open(meta_path_drive, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
if os.path.abspath(meta_path_drive) != os.path.abspath(meta_path_local):
    shutil.copy2(meta_path_drive, meta_path_local)
print(f"✓ Generated metadata: {meta_path_drive}")

# 3. Update nexus_data_backup.zip
target_zip = "/content/nexus_data_backup.zip" if os.path.exists("/content") else "./nexus_data_backup.zip"
print(f"Updating backup ZIP: {target_zip}...")
with zipfile.ZipFile(target_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(LOCAL_DIR):
        for file in files:
            if file.endswith(('.parquet', '.json', '.csv')):
                fpath = os.path.join(root, file)
                zipf.write(fpath, os.path.join("data", os.path.relpath(fpath, LOCAL_DIR)))
print(f"✓ Backup archive refreshed ({os.path.getsize(target_zip)/(1024*1024):.2f} MB)")

PHASE 14: PERSISTENCE & METADATA
✓ Saved dataset: /content/drive/MyDrive/NEXUS_Stock_AI/data/model_dataset_v1.parquet (10.21 MB)
✓ Generated metadata: /content/drive/MyDrive/NEXUS_Stock_AI/data/dataset_metadata.json
Updating backup ZIP: /content/nexus_data_backup.zip...
✓ Backup archive refreshed (276.03 MB)


### 🔍 Step 7: Final Validation Diagnostics & Dataset Preview

In [7]:
print("=" * 90)
print("🎯 FINAL VERIFICATION & CLASS BALANCE REPORT")
print("=" * 90)

# Class Balance Table
bal_df = pd.DataFrame({
    "Class": ["0 (DOWN / FLAT)", "1 (UP)"],
    "Count": [class_0_count, class_1_count],
    "Percentage": [
        f"{class_0_count / len(clean_model_df) * 100:.2f}%",
        f"{class_1_count / len(clean_model_df) * 100:.2f}%"
    ]
})
print("Class Distribution:")
display(bal_df)

# Dataset Overview
print("\nDataset Dimensionality:")
dim_df = pd.DataFrame({
    "Total Rows": [f"{len(clean_model_df):,}"],
    "Feature Columns": [len(feature_cols)],
    "Target Column": ["target"],
    "Unique Tickers": [clean_model_df['stock_symbol'].nunique()],
    "Date Range": [f"{clean_model_df['date'].min().strftime('%Y-%m-%d')} to {clean_model_df['date'].max().strftime('%Y-%m-%d')}"]
})
display(dim_df)

# Preview 5-row sample with key multi-modal features
print("\nMulti-Modal Dataset Sample (5 Rows):")
preview_cols = [
    'stock_symbol', 'date', 'close', 'return_1d', 'rsi_14',
    'news_count', 'sentiment_mean', 'positive_ratio', 'target'
]
display(clean_model_df[preview_cols].head(5))
print("=" * 90)
print("🎉 PHASE 6–14 DATASET ENGINEERING COMPLETE & READY FOR MODEL TRAINING!")

🎯 FINAL VERIFICATION & CLASS BALANCE REPORT
Class Distribution:


,Class,Count,Percentage
0,0 (DOWN / FLAT),33611,50.27%
1,1 (UP),33254,49.73%



Dataset Dimensionality:


,Total Rows,Feature Columns,Target Column,Unique Tickers,Date Range
0,"66,865",29,target,9,1980-05-27 to 2023-12-27



Multi-Modal Dataset Sample (5 Rows):


,stock_symbol,date,close,return_1d,rsi_14,news_count,sentiment_mean,positive_ratio,target
0,AAPL,1981-02-24,0.424107,-0.035533,30.302934,0,0.0,0.0,1
1,AAPL,1981-02-25,0.450893,0.063158,38.523747,0,0.0,0.0,1
2,AAPL,1981-02-26,0.457589,0.014851,40.415897,0,0.0,0.0,1
3,AAPL,1981-02-27,0.473214,0.034146,44.693362,0,0.0,0.0,1
4,AAPL,1981-03-02,0.475446,0.004717,45.297516,0,0.0,0.0,0


🎉 PHASE 6–14 DATASET ENGINEERING COMPLETE & READY FOR MODEL TRAINING!
